In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import re
import joblib
import string

In [40]:
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')

In [41]:
fake.head(4)

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"


In [42]:
true.head(4)

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"


In [43]:
fake['class']=0
true['class']=1

In [44]:
data=pd.concat([fake,true], axis=0)

In [45]:
data.sample(10)

,title,text,subject,date,class
13066,U.S. consulate in Jerusalem issues security me...,JERUSALEM (Reuters) - The U.S. Consulate in Je...,worldnews,"December 5, 2017",1
17476,ISRAEL WILL NAME New Train Station Near Wester...,Israel s transportation minister is pushing ah...,left-news,"Dec 27, 2017",0
10495,ERIC HOLDER Encourages DOJ To Keep Attacking T...,The most corrupt Attorney General in the histo...,politics,"Jul 1, 2017",0
9393,LOL! DESPERATE KATHY GRIFFIN Makes Home Video ...,Wow! Poor Kathy Griffin The toxic comedian f...,politics,"Nov 19, 2017",0
10373,Nancy Reagan remembered at funeral for fierce ...,"SIMI VALLEY, Calif. (Reuters) - First lady Mic...",politicsNews,"March 11, 2016",1
22408,Boiler Room EP #82 – Mind-boggling Collusion,Tune in to the Alternate Current Radio Network...,US_News,"November 3, 2016",0
9160,WHOA! PAUL RYAN Channels Trump…Blasts POLITICO...,"House Speaker Paul Ryan (R., Wis.) on Tuesday ...",politics,"Dec 19, 2017",0
13229,TRUMP’S WISCONSIN SPEECH Knocks It Out Of The ...,Donald Trump gave a rousing speech in Wisconsi...,politics,"Aug 17, 2016",0
3679,Clinton Campaign Announces BAD News For Trump...,"On Saturday morning, Hillary Clinton s campaig...",News,"November 26, 2016",0
15948,South Africa's unruly ANC branches kick off ra...,JOHANNESBURG (Reuters) - South Africa s ruling...,worldnews,"November 1, 2017",1


In [46]:
data = data.drop(['title', 'subject', 'date'], axis=1)

In [47]:
data.reset_index(inplace=True)

In [48]:
data.drop(['index'], axis=1, inplace=True)

In [49]:
data.sample(5)

,text,class
10608,Alex Jones scooped NBC and Megyn Kelly with a ...,0
1600,"Today, FBI Director James Comey sat down with ...",0
19353,The incident took place Tuesday in an apartmen...,0
22468,21st Century Wire says If you haven t seen Hil...,0
4488,There s already been hints from NBC News that ...,0


In [50]:
import re
import string

def clean_text(text):
    text = str(text)  # Ensure text is a string
    text = text.lower()

    # Remove text inside square brackets
    text = re.sub(r'\[.*?\]', '', text)

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>+', '', text)

    # Remove punctuation but keep spaces
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)

    # Remove newlines
    text = re.sub(r'\n', ' ', text)

    # Remove words containing digits
    text = re.sub(r'\w*\d\w*', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [51]:
sample = "This is FAKE news! Visit https://abc.com <p>Hello</p> 123"
print(clean_text(sample))

this is fake news visit hello


In [53]:
print(globals().keys())

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__vsc_ipynb_file__', '_i', '_ii', '_iii', '_i1', 'pd', '_i2', 'train_test_split', 'TfidfVectorizer', 'LogisticRegression', 'classification_report', 're', 'joblib', 'string', '_i3', 'fake', 'true', '_i4', '_i5', '_5', '_i6', '_6', '_i7', '_7', '_i8', '_8', '_i9', '_9', '_i10', '_i11', 'data', '_i12', '_12', '_i13', '_i14', '_i15', '_i16', '_16', '_i17', 'clean_text', '_i18', '_i19', '_i20', 'x', 'y', 'xtrain', 'xtest', 'ytrain', 'ytest', '_i21', 'vectorizer', '_i22', '_i23', '_i24', '_i25', 'i', '_i26', '_i27', '_i28', '_i29', '_i30', '_i31', '_i32', '_i33', 'sample', '_i34', '_i35', '_i36', '_i37', '_i38', '_i39', '_i40', '_i41', '_41', '_i42', '_42', '_i43', '_i44', '_i45', '_45', '_i46', '_i47', '_i48', '_i49', '_49', '_i50', '_i51', '_i52', '_i53'])


In [54]:
data['text'] = data['text'].fillna('')
data['text'] = data['text'].apply(clean_text)

In [55]:
data['text'] = data['text'].apply(clean_text)

In [56]:
print(data['text'].head())

0    donald trump just couldn t wish all americans ...
1    house intelligence committee chairman devin nu...
2    on friday it was revealed that former milwauke...
3    on christmas day donald trump announced that h...
4    pope francis used his annual christmas day mes...
Name: text, dtype: object


In [57]:
x = data['text']
y = data['class']

xtrain, xtest, ytrain, ytest = train_test_split(x,y,test_size=0.25, random_state=42)

In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
xv_train = vectorizer.fit_transform(xtrain)
xv_test = vectorizer.transform(xtest)

print(xv_train.shape)

(33673, 93866)


In [59]:
lr = LogisticRegression()
lr.fit(xv_train, ytrain)

LogisticRegression()

In [60]:
prediction = lr.predict(xv_test)
lr.score(xv_test, ytest)

0.9860133630289533

In [61]:
print(classification_report(ytest, prediction))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5895
           1       0.98      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [64]:
joblib.dump(vectorizer, 'vectorizer.jb')
joblib.dump(lr, 'lr_model.jb')

['lr_model.jb']